In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from catboost import CatBoostClassifier

print("=== ЗАПУСК CUSTOMER ANALYTICS PIPELINE ===")

# ==========================================================
# СЛОЙ 1: АНАЛИТИКА, КЛАСТЕРИЗАЦИЯ И ПРОГНОЗИРОВАНИЕ (ML)
# ==========================================================
np.random.seed(42)
n_customers = 2000

customer_ids = np.arange(20001, 20001 + n_customers)
recency = np.random.randint(1, 365, size=n_customers)
frequency = np.random.randint(1, 30, size=n_customers)
monetary = (frequency * np.random.normal(1500, 300, size=n_customers)).round(2)

churn_risk = (recency * 0.7 - monetary * 0.001)
churn_labels = (churn_risk > np.percentile(churn_risk, 80)).astype(int)

bi_table = pd.DataFrame({
    'CustomerID': customer_ids,
    'Recency': recency,
    'Frequency': frequency,
    'Monetary': monetary,
    'IsChurn': churn_labels
})

scaler = StandardScaler()
scaled_data = scaler.fit_transform(bi_table[['Recency', 'Frequency', 'Monetary']])
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
bi_table['Segment'] = kmeans.fit_predict(scaled_data)

X = bi_table[['Recency', 'Frequency', 'Monetary', 'Segment']]
y = bi_table['IsChurn']

ctb_model = CatBoostClassifier(iterations=20, learning_rate=0.1, verbose=0)
ctb_model.fit(X, y)

print(f"ML-Слой успешно обработал {n_customers} профилей клиентов.")
print(f"Выявлено потенциальных клиентов на уход: {bi_table['IsChurn'].sum()}")

# ==========================================================
# СЛОЙ 2: КОНВЕЙЕР ИИ-КОНТЕНТА (Генерация офферов)
# ==========================================================
target_segments = {
    0: "VIP-клиент (высокий чек, регулярные покупки). Стратегия ИИ: Удержать эксклюзивным закрытым предложением.",
    1: "Критический статус (риск оттока). Стратегия ИИ: Вернуть любой ценой, сгенерировать скидку 50%.",
    2: "Новый клиент (низкий чек, одна покупка). Стратегия ИИ: Прогреть контентом на повторную покупку."
}

def generate_ai_campaign(customer_row):
    segment_id = customer_row['Segment']
    strategy = target_segments.get(segment_id, "Стандартная кампания")
    
    html_email_template = f"""
    <div style='font-family: Arial, sans-serif; padding: 20px; border: 2px solid #4169E1; border-radius: 8px; max-width: 500px;'>
        <h3 style='color: #4169E1;'>Персонализированная кампания для Клиента #{int(customer_row['CustomerID'])}</h3>
        <p><b>Аналитика трат:</b> Сформированный оборот {customer_row['Monetary']} руб.</p>
        <p><b>Логика ИИ-Агента:</b> {strategy}</p>
        <hr style='border: 0; border-top: 1px solid #eee;' />
        <p style='font-size: 12px; color: #666;'>Сгенерировано автоматически ИИ-пайплайном на основе предикта CatBoost.</p>
    </div>
    """
    return html_email_template

# Тестируем на уходящем клиенте
critical_customer = bi_table[bi_table['IsChurn'] == 1].iloc[0]
generated_html = generate_ai_campaign(critical_customer)

print("\n=== РЕЗУЛЬТАТ РАБОТЫ ИИ-КОНВЕЙЕРА ===")
print(generated_html)


=== ЗАПУСК CUSTOMER ANALYTICS PIPELINE ===
ML-Слой успешно обработал 2000 профилей клиентов.
Выявлено потенциальных клиентов на уход: 400

=== РЕЗУЛЬТАТ РАБОТЫ ИИ-КОНВЕЙЕРА ===

    <div style='font-family: Arial, sans-serif; padding: 20px; border: 2px solid #4169E1; border-radius: 8px; max-width: 500px;'>
        <h3 style='color: #4169E1;'>Персонализированная кампания для Клиента #20002</h3>
        <p><b>Аналитика трат:</b> Сформированный оборот 14211.32 руб.</p>
        <p><b>Логика ИИ-Агента:</b> Критический статус (риск оттока). Стратегия ИИ: Вернуть любой ценой, сгенерировать скидку 50%.</p>
        <hr style='border: 0; border-top: 1px solid #eee;' />
        <p style='font-size: 12px; color: #666;'>Сгенерировано автоматически ИИ-пайплайном на основе предикта CatBoost.</p>
    </div>
    


In [2]:
from IPython.display import display, HTML

# Передаем сгенерированный HTML-код в модуль отображения
display(HTML(generated_html))
